In [1]:
import wrds
conn = wrds.Connection(wrds_username='muhab')

# Check the absolute last date in CRSP
latest = conn.raw_sql("""
    SELECT MAX(date) as last_date
    FROM crsp.msf
""")
print("Last date in CRSP:", latest['last_date'].values[0])

WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Last date in CRSP: 2024-12-31


In [2]:
import pandas as pd

crsp = pd.read_csv('crsp_returns.csv', parse_dates=['date'])

extension = crsp[crsp['date'] >= '2022-01-01']
print(f"Extension period rows: {len(extension):,}")
print(f"Date range: {extension['date'].min()} to {extension['date'].max()}")
print(f"Unique permnos: {extension['permno'].nunique():,}")

Extension period rows: 213,972
Date range: 2022-01-01 00:00:00 to 2024-11-01 00:00:00
Unique permnos: 7,258


In [4]:
# First check what Compustat's last date is
latest_comp = conn.raw_sql("""
    SELECT MAX(datadate) as last_date
    FROM comp.funda
    WHERE indfmt = 'INDL'
    AND datafmt = 'STD'
    AND popsrc = 'D'
    AND consol = 'C'
""")
print("Last Compustat date:", latest_comp['last_date'].values[0])

Last Compustat date: 2026-02-28


In [5]:
# Step 1: Get the permnos from your extension period
extension_permnos = crsp[crsp['date'] >= '2022-01-01']['permno'].unique().tolist()
print(f"Permnos in extension period: {len(extension_permnos):,}")

# Step 2: Pull link table for these permnos
def pull_link_table(conn, permnos):
    permno_str = ','.join(map(str, permnos))
    
    print("Pulling CCM link table...")
    link = conn.raw_sql(f"""
        SELECT
            gvkey,
            lpermno AS permno,
            linktype,
            linkprim,
            linkdt,
            linkenddt
        FROM crsp.ccmxpf_lnkhist
        WHERE linktype IN ('LC', 'LU')
            AND linkprim IN ('P', 'C')
            AND lpermno IN ({permno_str})
    """, date_cols=['linkdt', 'linkenddt'])
    
    link['linkenddt'] = link['linkenddt'].fillna(pd.Timestamp('2099-12-31'))
    
    print(f"Link rows: {len(link):,}")
    print(f"Unique gvkeys: {link['gvkey'].nunique():,}")
    print(f"Unique permnos matched: {link['permno'].nunique():,}")
    
    return link

link = pull_link_table(conn, extension_permnos)
link.to_csv('link_table.csv', index=False)
print("Saved: link_table.csv")

Permnos in extension period: 7,258
Pulling CCM link table...
Link rows: 7,254
Unique gvkeys: 6,776
Unique permnos matched: 6,370
Saved: link_table.csv


In [6]:
def pull_compustat(conn, gvkeys, start='1950-01-01', end='2024-12-31'):
    """
    Start from 1950 even though extension is 2022-2024.
    We need historical data to compute lagged variables like:
    - asset growth (needs prior year assets)
    - momentum (needs prior year returns)
    - ROA improvement (needs prior year ROA)
    Without history, these features will be NaN for early rows.
    """
    
    # Pull in chunks to avoid timeout
    chunk_size = 500
    chunks = [gvkeys[i:i+chunk_size] 
              for i in range(0, len(gvkeys), chunk_size)]
    
    all_chunks = []
    
    for i, chunk in enumerate(chunks):
        print(f"  Pulling chunk {i+1}/{len(chunks)}...")
        gvkey_str = ','.join([f"'{g}'" for g in chunk])
        
        chunk_df = conn.raw_sql(f"""
            SELECT
                gvkey,
                datadate,
                fyear,
                fyr,
                -- Balance sheet
                at, lt, seq, ceq,
                pstk, pstkrv, pstkl,
                txditc, txp,
                act, lct, che, rect, invt,
                dlc, dltt,
                -- Income statement
                ni, ib, oiadp, oibdp,
                dp, sale, cogs, xsga, xrd,
                -- Cash flow
                capx,
                -- Other
                dv, csho, prcc_f,
                naicsh, sich
            FROM comp.funda
            WHERE datadate BETWEEN '{start}' AND '{end}'
                AND gvkey IN ({gvkey_str})
                AND indfmt = 'INDL'
                AND datafmt = 'STD'
                AND popsrc = 'D'
                AND consol = 'C'
                AND at > 0
        """, date_cols=['datadate'])
        
        all_chunks.append(chunk_df)
    
    comp = pd.concat(all_chunks, ignore_index=True)
    
    # Remove duplicates
    comp = comp.sort_values(['gvkey', 'datadate'])
    comp = comp.drop_duplicates(subset=['gvkey', 'datadate'], keep='last')
    
    print(f"\nCompustat rows: {len(comp):,}")
    print(f"Unique firms: {comp['gvkey'].nunique():,}")
    print(f"Date range: {comp['datadate'].min()} to {comp['datadate'].max()}")
    
    return comp

gvkeys = link['gvkey'].unique().tolist()
print(f"Pulling Compustat for {len(gvkeys)} gvkeys...")

comp = pull_compustat(conn, gvkeys)
comp.to_csv('compustat_annual.csv', index=False)
print("Saved: compustat_annual.csv")

conn.close()

Pulling Compustat for 6776 gvkeys...
  Pulling chunk 1/14...
  Pulling chunk 2/14...
  Pulling chunk 3/14...
  Pulling chunk 4/14...
  Pulling chunk 5/14...
  Pulling chunk 6/14...
  Pulling chunk 7/14...
  Pulling chunk 8/14...
  Pulling chunk 9/14...
  Pulling chunk 10/14...
  Pulling chunk 11/14...
  Pulling chunk 12/14...
  Pulling chunk 13/14...
  Pulling chunk 14/14...

Compustat rows: 123,085
Unique firms: 5,931
Date range: 1950-06-30 00:00:00 to 2024-12-31 00:00:00
Saved: compustat_annual.csv
